# Execution 研究

通过插件将上游项目安装为 dev 依赖。表单构造本项目的 `ExecutionAnalysisParams`，继承当前算法的 `ExecutionParams`。只在此 Notebook 合并上游参数，算法包不依赖上游项目。Scheme 各环节读取统一的同名字段。

In [ ]:
%load_ext autoreload
%autoreload 2

from importlib import import_module
from execution import ExecutionAlgo, ExecutionAnalysisParams, ExecutionReportForm
from scheme import StockPool, algo_options
from scheme.execute.strategy.assembly import parameter_type, project_parameters

model_options = algo_options("model")
optimize_options = algo_options("optimize")
control_options = algo_options("control")

model_options, optimize_options, control_options

## 选择与配置上游
先查看上一格的可用入口，然后显式填写下方的 `*_entry`，即使只有一个选项也不会自动选中。没有选项时请先安装兼容上游；多个选项时由研究者决定。`*_overrides` 只填写算法特有字段（如 Model 的 `n_select`），不要覆盖公共研究设置。`start`、`end`、`universe` 等从当前分析参数传播给上游；参数类型从选中的 Algo 泛型解析，无需假定包内导出的 Params 名称。使用 Scheme 1.2.1+ 的参数投影辅助函数。

In [ ]:
def selected_upstream(entry):
    module, name = entry.split(":")
    return parameter_type(getattr(import_module(module), name))


def merge_upstream(analysis, upstream):
    values = dict(analysis)  # 保留嵌套 Universe。
    for parameter_model, overrides in upstream:
        values.update(dict(project_parameters(parameter_model, analysis, overrides=overrides)))
    return type(analysis).model_validate(values)


In [ ]:
model_entry = None  # 显式填入 model_options 中的入口字符串
model_overrides = {}  # 例如 {"n_select": 10}，仅算法特有字段
optimize_entry = None  # 显式填入 optimize_options 中的入口字符串
optimize_overrides = {}  # 仅该算法声明的非公共字段
control_entry = None  # 显式填入 control_options 中的入口字符串
control_overrides = {}  # 仅该算法声明的非公共字段

ModelParams = selected_upstream(model_entry)
OptimizeParams = selected_upstream(optimize_entry)
ControlParams = selected_upstream(control_entry)

form = ExecutionReportForm(
    start="2026-06-01",
    end="2026-06-06",
    pool=StockPool.SSE50,
    model=model_entry,
    optimize=optimize_entry,
    control=control_entry,
)


In [ ]:
params: ExecutionAnalysisParams = form.build()
params = merge_upstream(params, [
    (ModelParams, model_overrides),
    (OptimizeParams, optimize_overrides),
    (ControlParams, control_overrides),
])
params

In [ ]:
report = params.run(ExecutionAlgo)
report.show()